# 00 – EDA & Feature Store Story

This notebook is a **narrative scaffold** for exploring the Kaggle Online Payments Fraud Detection Dataset and mapping it into a Feast-based feature store.

The goal is to:
- Understand the fraud problem and why feature stores help.
- Perform initial EDA (imbalance, transaction types, amounts).
- Design features and metrics aligned with `context/04_DATASET_PLAN.md` and `context/05_METRICS_AND_EVAL.md`.
- Prepare a story that explains how offline analysis turns into online Feast features.

## 1. Problem framing & why feature stores

Topics to cover:
- Nature of online payments fraud (rare, adversarial, time-sensitive).
- Challenges with ad-hoc feature pipelines (leakage, inconsistency, duplication).
- How a feature store (Feast) can help:
  - Centralized feature definitions.
  - Unified offline/online access.
  - Clear separation between feature engineering and modeling.
- How this project will use Feast with Postgres + Kafka/Redpanda.

In [ ]:
# TODO: Import core libraries and set up paths
# import pandas as pd
# import numpy as np
# from pathlib import Path

# DATA_DIR = Path("../data/raw")
# Example: df = pd.read_csv(DATA_DIR / "onlinefraud.csv")

## 2. Dataset overview (Kaggle) & leakage checks

Topics to cover:
- Columns and data types:
  - `step`, `type`, `amount`, `nameOrig`, `oldbalanceOrg`, `newbalanceOrig`,
    `nameDest`, `oldbalanceDest`, `newbalanceDest`, `isFraud`, `isFlaggedFraud`.
- Basic descriptive statistics (counts, missing values, ranges).
- Leakage checks:
  - Are there columns that include post-transaction outcomes?
  - Are there columns that effectively encode the target (`isFraud`)?
- Time aspect of `step` and its implications for train/validation/test splits.

In [ ]:
# TODO: Load dataset and perform initial schema inspection
# df = pd.read_csv(DATA_DIR / "onlinefraud.csv")
# df.head()
# df.info()
# df.describe(include="all")

## 3. EDA: target imbalance, transaction types, amounts

Topics to cover:
- Distribution of `isFraud` and `isFlaggedFraud`.
- Relationship between `type` and fraud rate.
- Distribution of `amount` (overall and by `type`).
- Simple temporal plots over `step` (volume and fraud rate).

This section should justify why metrics like PR-AUC and recall@precision are important (see `context/05_METRICS_AND_EVAL.md`).

In [ ]:
# TODO: Basic EDA for target imbalance and transaction types
# df["isFraud"].value_counts(normalize=True)
# df.groupby("type")["isFraud"].mean().sort_values(ascending=False)
# df["amount"].describe()

## 4. Feature engineering rationale (velocity, balances, ratios)

Topics to cover:
- Velocity features:
  - Number and total amount of transactions per entity over sliding windows.
- Balance and ratio features:
  - `amount / (oldbalanceOrg + 1)`, `oldbalanceOrg - newbalanceOrig`, etc.
- Counterparty interaction features:
  - Frequency and recency of transfers between `nameOrig` and `nameDest`.
- Justify which features are:
  - Feasible online (real-time or near real-time).
  - Robust to free-tier infrastructure constraints.

In [ ]:
# TODO: Prototype a few engineered features and inspect them
# df["amount_ratio"] = df["amount"] / (df["oldbalanceOrg"] + 1)
# df[["amount", "amount_ratio"]].head()

## 5. Baseline model & metric choice rationale (why PR-AUC)

Topics to cover:
- Define a simple baseline model (e.g., logistic regression, tree-based model).
- Explain the train/validation/test split strategy (time-aware).
- Compute:
  - PR-AUC
  - ROC-AUC
  - Recall@precision and Precision@recall thresholds
- Explain why PR-AUC is preferred for this imbalanced setting (link to `context/05_METRICS_AND_EVAL.md`).

In [ ]:
# TODO: Train a simple baseline model and compute evaluation metrics
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import average_precision_score, roc_auc_score
# from sklearn.linear_model import LogisticRegression

# X = ...  # engineered features
# y = df["isFraud"]
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
# model = LogisticRegression(max_iter=1000)
# model.fit(X_train, y_train)
# y_score = model.predict_proba(X_test)[:, 1]
# pr_auc = average_precision_score(y_test, y_score)
# roc_auc = roc_auc_score(y_test, y_score)
# pr_auc, roc_auc

## 6. Feature importance (planned: permutation / SHAP later)

Topics to cover:
- How we might assess feature importance:
  - Model-specific importance (e.g., Gini importance for trees).
  - Permutation importance.
  - SHAP values.
- How interpretability influences feature and model choices.
- How to align feature importance analysis with the Feast feature definitions.

In [ ]:
# TODO: Placeholder for feature importance analysis
# Example (to be filled later): permutation importance or SHAP summary plots

## 7. Mapping features -> Feast FeatureViews

Topics to cover:
- For each engineered feature, specify:
  - Entity (customer_id, account_id, counterparty_id, device_id, geo_cell_id).
  - Source table or stream (batch file, Postgres table, Kafka topic).
  - Freshness / TTL requirements.
- Sketch how these features will be represented in Feast:
  - `Entity` definitions.
  - `FeatureView` definitions.
  - Materialization strategies (batch + streaming).

The goal is to make this notebook a bridge between data exploration and concrete Feast configuration in `feature_repo/`.

In [ ]:
# TODO: Draft a mapping table (e.g., as a DataFrame) linking features to Feast constructs
# feature_catalog = pd.DataFrame([
#     {"feature_name": "amount_ratio", "entity": "account_id", "source": "batch", "ttl": "7d"},
# ])
# feature_catalog

## 8. Online serving considerations & latency measurement plan

Topics to cover:
- How features will be retrieved online (via Feast + Postgres).
- Latency budget breakdown:
  - Network latency HF Spaces ↔ Postgres/Kafka.
  - Feature retrieval time.
  - Model inference time.
  - Total request latency budget (p95, p99).
- How Prometheus metrics and logs will be used to track latency and reliability.
- Plans for experimentation with caching and pre-computation if needed.

In [ ]:
# TODO: Sketch code to measure request and feature retrieval latency
# For example, wrappers around Feast feature retrieval + model inference that record timing
# and emit metrics usable by Prometheus.